# 03 · Vector Databases

> **Source notes:** `VectorDBs.md` + `VectorDBs_Supplement.md`

How does a vector store find the nearest neighbours in milliseconds across millions of vectors?

This notebook walks through:
- Distance metrics (L2, Cosine, Dot Product)
- Why brute-force search fails at scale
- IVF clustering-based ANN index
- HNSW graph-based ANN index (used by ChromaDB, Pinecone, Qdrant, Weaviate)
- DiskANN and quantization concepts
- Practical ChromaDB demo

**Tools (all local):** `numpy`, `scikit-learn`, `chromadb`, `sentence-transformers`

In [ ]:
# TODO: Implement this cell


## 1 · Distance Metrics — How "Closeness" Is Measured

All vector search rests on a **distance (or similarity) function**:

| Metric | Formula | Use When |
|--------|---------|----------|
| **Euclidean (L2)** | `||a - b||` | Image embeddings, sensor data |
| **Cosine Similarity** | `(a · b) / (||a|| · ||b||)` | Text / semantic similarity |
| **Dot Product** | `a · b` | Recommendation; magnitude matters |

**Interview-critical fact:** when vectors are **L2-normalised** (unit length), cosine similarity and dot product give identical rankings, and maximising dot product equals minimising Euclidean distance.

Most embedding models (sentence-transformers with `normalize_embeddings=True`) output unit vectors, making the metric choice less critical in practice — but knowing *why* matters.

In [ ]:
# TODO: Implement this cell


## 2 · Why Brute-Force Search Fails at Scale

**Exact search** = compute distance to **every** stored vector.

**Time complexity:** `O(N x d)` per query.

| Scale | Dimensions | Ops/query | Time at 10 GFLOP/s |
|-------|-----------|-----------|---------------------|
| 100 K | 384 | 38 M | ~4 ms OK |
| 10 M | 768 | 7.7 B | ~770 ms TOO SLOW |
| 100 M | 768 | 77 B | ~7.7 s WAY TOO SLOW |

**Memory:** 100 M vectors x 768 dims x 4 bytes (float32) = **307 GB** — exceeds most server RAM.

Traditional indexes (kd-trees, ball-trees) degrade in high dimensions — "the curse of dimensionality."

**Solution: Approximate Nearest Neighbour (ANN) indexes** — trade a tiny bit of recall for massive speed gains.

In [ ]:
# TODO: Implement this cell


## 3 · IVF — Inverted File Index

**Idea:** partition the vector space into **K clusters** (k-means). At query time only search the closest `nprobe` clusters instead of everything.

```
Training:
  K-means on corpus --> K centroids + N inverted lists

Query:
  1. Compute distance from query to all K centroids  (cheap: O(K * d))
  2. Pick nprobe closest centroids
  3. Search only the vectors in those clusters
```

**Key tunables:**
- `K` — number of clusters (more clusters = finer partitioning = faster, but lower recall per cluster)
- `nprobe` — clusters searched at query time (higher = better recall, slower)

In [ ]:
# TODO: Implement this cell


## 4 · HNSW — Hierarchical Navigable Small World

HNSW is a **graph-based** ANN index used by most production vector DBs (ChromaDB, Pinecone, Qdrant, Weaviate, Azure AI Search).

```
Layer 2 (sparse long-range links)   A ----------- B
                                    |             |
Layer 1                             A --- C --- D-B
                                    |             |
Layer 0 (dense local links)         A-c-C-d-D-e-E-B  <- most vectors live here
```

**How search works:**
1. Enter at a random node in the **top (sparse) layer**
2. Greedily navigate toward the query (always step to the closer neighbour)
3. Drop to the layer below when stuck; repeat
4. At Layer 0, do a local exhaustive search among found neighbours

**Key parameters:**
- `M` — max edges per node (higher = better recall, more memory, slower build)
- `ef_construction` — size of dynamic list during build (higher = better quality graph)
- `ef` — size of candidate list at query time (higher = better recall, slower query)

ChromaDB uses HNSW internally with sensible defaults. We just configure the space (metric).

In [ ]:
# TODO: Implement this cell


## 5 · DiskANN, Quantization & Index Selection

### DiskANN
- Stores the HNSW graph primarily on **SSD**, caching hot nodes in RAM
- Enables billion-scale search on commodity servers
- Used in Azure AI Search

### Product Quantization (PQ)
Compress vectors by splitting each high-dim vector into M sub-vectors, each quantised to a codebook:

| Original | PQ compressed | Compression |
|---------|--------------|-------------|
| 768 × 4 B = 3072 B | ~96 B | ~32x |

Trade-off: memory savings at the cost of some recall loss.

### Index Selection Guide

| Scenario | Recommended |
|----------|------------|
| < 100 K vectors, perfect recall | **Flat** (brute-force) |
| 100 K – 10 M, general purpose | **HNSW** |
| Large corpus, memory-constrained | **IVF + PQ** |
| Billions of vectors, commodity hardware | **DiskANN** |
| Streaming inserts (no rebuild) | **HNSW** (supports incremental adds) |

## 6 · Key Takeaways

| Concept | One-Liner |
|---------|-----------|
| L2 / Cosine / Dot | On normalised vectors they give identical rankings |
| Brute-force | Perfect recall; O(N*d) cost; impractical above ~100 K |
| IVF | K-means partitioning; tune nprobe for recall vs. speed |
| HNSW | Graph-based greedy descent; dominant in production |
| DiskANN | HNSW for billion-scale; data lives on SSD |
| PQ | Compress vectors 32x; small recall loss |

**Next:** `04-react-agents/notebook.ipynb` — building an agent that uses all of this.

## 6 · PizzaBot Connection — Index Choice for a Small Corpus

> Full system spec: [AIPrimer.md](../AIPrimer.md)

The PizzaBot menu corpus (~500 chunks) is small enough that the index choice is mostly invisible to users — but it makes the tradeoffs concrete:

| Scenario | Index | Why |
|---|---|---|
| **Prototype** | FAISS Flat (brute-force, exact) | 500 vectors; exact search < 1 ms; no tuning needed |
| **10k menu items** | FAISS `IndexHNSWFlat` | Sub-millisecond search; `efSearch=64` balances recall/latency |
| **Multi-store (1M+ items)** | IVF-PQ in a dedicated vector DB | Corpus distribution across stores matches IVF cluster assumptions |
| **Allergen safety queries** | Flat or `efSearch=128+` | Safety-critical: favour recall over latency — never miss a allergen flag |

**Key insight applied here:** cosine similarity is the right metric for MiniLM embeddings because they are not L2-normalised by default. The FAISS L2 metric would give different (incorrect) results.

**Try it:** add a FAISS cell that embeds the PizzaBot `allergens.csv` rows and queries `"dairy-free options"` — observe that it returns items without dairy even when the query phrasing doesn't match any row verbatim.
